In [8]:
t = @task begin; sleep(5);println("done");end

Task (runnable) @0x00000001121dc010

In [9]:
schedule(t);

In [14]:
Threads.@spawn begin;sleep(5);println("end spawn") end
sin(pi/4)

0.7071067811865475

end spawn


In [15]:
Threads.nthreads()

4

In [ ]:
# Start Julia with multiple threads, e.g., `julia --threads 4`

function process_data_pipeline()
    # 1. Create a Channel (our conveyor belt) that can hold 10 items at a time
    jobs = Channel{Int}(10)
    
    # 2. THE PRODUCER (I/O Bound)
    # We use @async because fetching data is just waiting on the network/disk.
    # It doesn't need its own CPU core.
    @async begin
        println("Producer: Starting to fetch records...")
        for i in 1:20
            sleep(0.1) # Simulate I/O latency (e.g., waiting for database)
            put!(jobs, i) # Put the fetched record onto the conveyor belt
        end
        close(jobs) # Tell the workers no more data is coming
        println("Producer: Finished fetching all records.")
    end

    # 3. THE CONSUMERS (CPU Bound)
    # We use @sync to wait for all the spawned worker threads to finish
    @sync begin
        # Spawn exactly as many workers as we have CPU threads available
        for worker_id in 1:Threads.nthreads()
            # Threads.@spawn puts this Task on any available CPU core
            Threads.@spawn begin
                # take! automatically waits if the channel is empty, 
                # and safely grabs the next item without race conditions
                for record in jobs
                    println("Worker $worker_id: Crunching numbers for record $record on thread $(Threads.threadid())")
                    
                    # Simulate heavy CPU computation (e.g., matrix math)
                    # Notice we don't use sleep() here, we simulate real CPU work
                    sum = 0
                    for _ in 1:10_000_000
                        sum += 1
                    end
                end
                println("Worker $worker_id: Shutting down.")
            end
        end
    end
    
    println("Pipeline complete!")
end

process_data_pipeline()

Producer: Starting to fetch records...
Worker 4: Crunching numbers for record 1 on thread 2
Worker 4: Crunching numbers for record 2 on thread 2
Worker 2: Crunching numbers for record 3 on thread 3
Worker 4: Crunching numbers for record 4 on thread 2
Worker 4: Crunching numbers for record 5 on thread 2
Worker 4: Crunching numbers for record 6 on thread 2
Worker 4: Crunching numbers for record 7 on thread 2
Worker 2: Crunching numbers for record 8 on thread 3
Worker 4: Crunching numbers for record 9 on thread 2
Worker 4: Crunching numbers for record 10 on thread 2
Worker 4: Crunching numbers for record 11 on thread 2
Worker 4: Crunching numbers for record 12 on thread 2
Worker 2: Crunching numbers for record 13 on thread 3
Worker 4: Crunching numbers for record 14 on thread 2
Worker 4: Crunching numbers for record 15 on thread 2
Worker 2: Crunching numbers for record 16 on thread 3
Worker 4: Crunching numbers for record 17 on thread 2
Worker 4: Crunching numbers for record 18 on thread 

done
end spawn


In [18]:
Threads.@threads for i in 1:10
    println("Processing item $i on thread $(Threads.threadid())")
end

Processing item 7 on thread 3
Processing item 8 on thread 3
Processing item 4 on thread 4
Processing item 9 on thread 2
Processing item 1 on thread 5
Processing item 5 on thread 4
Processing item 6 on thread 4
Processing item 2 on thread 5
Processing item 3 on thread 5
Processing item 10 on thread 2
